**Important: read the following before you hit "Run all"**

Model checkpoints will be saved and loaded locally.

This notebook uses Google Colab. So, local means local disk used in the current Google Colab session.

To persist model checkpoints across sessions, this notebook saves the local copy to Google Drive after training.

To load a model back, this notebook copies model checkpoints from Google Drive back to Google Colab.

To transfer data between Google Colab and Google Drive, Google Drive is mounted under "/content/drive" and our data can be accessed via "/content/drive/MyDrive" in Google Colab.

**Checkpoints needed for training:**

R1 checkpoints on Google Drive: /checkpoints/r1_checkpoints

**Dataset needed for training**

Preference pairs dataset on Google Drive: /datasets/r2_preference_dataset

**Final model saved after training:**

R2 on Google Drive: /checkpoints/r2_checkpoints/r2-dpo-best

**Note**:

1) Change these locations and files operations to match your environments as necessary. For example, you may mount your Google Drive, then load and save checkpoints directly from Google Drive. Though, load & save on local + copy before load & save after training between Google Colab and Google Drive as described above is preferred.

2) Multiple checkpoints will be saved during training, which will consume a lot of disk space. Therefore, unless we want these checkpoints for resumption or continuous training across Google Colab sessions, we should only copy the final model from Google Colab to Google Drive. Otherwise, we may find out it would eat up our Google Drive space pretty fast.

In [ ]:
# First, mount Drive (run in a cell)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy R1 & Preference Dataset from Google Drive to local

# For the first run
!cp -r drive/MyDrive/checkpoints/r1_checkpoints/ .

# For the first run
!cp -r drive/MyDrive/datasets/r2_preference_dataset .

# Beyond the first run
# Make a folder for R2 checkpoints
!mkdir r2_checkpoints
!cp -r drive/MyDrive/checkpoints/r2_checkpoints/r2-dpo-best r2_checkpoints


In [ ]:
# Install TRL
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 14.4 MB/s  0:00:00


In [ ]:
import os
import json
import torch
import transformers
import trl

# Check dependencies

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Transformers: 5.22.0
TRL: 1.17.1
CUDA available: True
GPU: Tesla T4


In [ ]:
# Imports and configuration

# This implementation uses the current TRL DPOConfig / DPOTrainer API

from pathlib import Path

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
)
from transformers.trainer_utils import get_last_checkpoint

from trl import DPOConfig, DPOTrainer

# --------------------------------------------------
# Model paths
# --------------------------------------------------

R1_CHECKPOINT = "./r1_checkpoints/r1-sft-best"

# R1 remains the fixed reference model throughout R2.
REFERENCE_MODEL_PATH = R1_CHECKPOINT

# The initial policy model for this first DPO run.
# INITIAL_POLICY_PATH = R1_CHECKPOINT



# --------------------------------------------------
# Dataset paths
# --------------------------------------------------

DATASET_DIR = Path("./r2_preference_dataset")

TRAIN_DATASET_PATH = DATASET_DIR / "train.jsonl"
VALIDATION_DATASET_PATH = DATASET_DIR / "validation.jsonl"


# --------------------------------------------------
# Run/checkpoint paths
# --------------------------------------------------

CHECKPOINT_ROOT = Path("./r2_checkpoints")

# Use run_01 for the first training run.
# RUN_NAME = "run_01"
# Beyond the first training run
RUN_NAME = "run_02"

# Trainer saves checkpoint-50, checkpoint-100, etc. here.
OUTPUT_DIR = CHECKPOINT_ROOT / "runs" / RUN_NAME

# Standalone best model, loaded for future experiments.
BEST_MODEL_DIR = CHECKPOINT_ROOT / "r2-dpo-best"

# Beyond the first training run, use the best model previously trained
INITIAL_POLICY_PATH = str(BEST_MODEL_DIR)

# --------------------------------------------------
# Training configuration
# --------------------------------------------------

SEED = 42

NUM_TRAIN_EPOCHS = 3

PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4

GRADIENT_ACCUMULATION_STEPS = 2

LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.01
#WARMUP_RATIO is deprecated and replaced with WARMUP_STEPS
#WARMUP_RATIO = 0.05
# WARMUP_STEPAS is calculated as:
# #training_examples / batch_size / gradient_accumu_steps * #epochs * warmup_ratio
WARMUP_STEPS = 22

DPO_BETA = 0.1

MAX_LENGTH = 160
MAX_PROMPT_LENGTH = 120

LOGGING_STEPS = 5
EVAL_SAVE_STEPS = 50

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

USE_FP16 = torch.cuda.is_available()

print("Device:", DEVICE)
print("FP16:", USE_FP16)
print("Output directory:", OUTPUT_DIR)

Device: cuda
FP16: True
Output directory: r2_checkpoints/runs/run_02


In [ ]:
# Load and validate preference datasets
# We only need the three DPO fields.
# (The diagnostic columns from our dataset-generation process are not needed for optimization.)

assert TRAIN_DATASET_PATH.exists(), (
    f"Training dataset not found: {TRAIN_DATASET_PATH}"
)

assert VALIDATION_DATASET_PATH.exists(), (
    f"Validation dataset not found: {VALIDATION_DATASET_PATH}"
)

dataset_dict = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_DATASET_PATH),
        "validation": str(VALIDATION_DATASET_PATH),
    },
)

train_dataset = dataset_dict["train"].select_columns(
    ["prompt", "chosen", "rejected"]
)

validation_dataset = dataset_dict["validation"].select_columns(
    ["prompt", "chosen", "rejected"]
)

print("Training pairs:", len(train_dataset))
print("Validation pairs:", len(validation_dataset))

print("\nFirst training example:")
print(train_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Training pairs: 1178
Validation pairs: 215

First training example:
{'prompt': 'Sentiment: positive\nContinue the following review:\nthat loves its characters and communicates something', 'chosen': ' deeply personal --', 'rejected': ' deeply personal about them , even as a child'}


In [ ]:
# Perform basic data validation

def validate_preference_dataset(dataset, name):
    for i, example in enumerate(dataset):
        for field in ["prompt", "chosen", "rejected"]:
            value = example[field]

            assert isinstance(value, str), (
                f"{name}[{i}]: {field} must be a string"
            )

            assert len(value.strip()) > 0, (
                f"{name}[{i}]: {field} must not be empty"
            )

        assert example["chosen"] != example["rejected"], (
            f"{name}[{i}]: chosen and rejected are identical"
        )

    print(f"{name}: validation passed")


validate_preference_dataset(train_dataset, "Training")
validate_preference_dataset(validation_dataset, "Validation")

Training: validation passed
Validation: validation passed


In [ ]:
# Load tokenizer, policy model, and reference model
""" This is a particularly important part of DPO.

We load two independent GPT-2 instances from R1 (the same checkpoints):
- policy_model: trainable.
- ref_model: frozen.
Initially, their parameters are identical.

Why explicitly load the reference model?

TRL can construct reference behavior automatically in some configurations,
but for this educational experiment I prefer making the two models explicit.
It makes the role of the reference policy easier to understand and inspect.
"""

tokenizer = AutoTokenizer.from_pretrained(
    R1_CHECKPOINT
)

# GPT-2 has no dedicated padding token.
tokenizer.pad_token = tokenizer.eos_token

# TRL DPOTrainer expects left padding for batched causal LM inputs.
tokenizer.padding_side = "left"


# Trainable policy.
policy_model = GPT2LMHeadModel.from_pretrained(
    INITIAL_POLICY_PATH
)

policy_model.config.pad_token_id = tokenizer.pad_token_id
policy_model.config.use_cache = False


# Frozen reference.
ref_model = GPT2LMHeadModel.from_pretrained(
    REFERENCE_MODEL_PATH
)

ref_model.config.pad_token_id = tokenizer.pad_token_id
ref_model.config.use_cache = False

for parameter in ref_model.parameters():
    parameter.requires_grad = False

# We should see approximately 124 million parameters for each model, with all policy parameters trainable and all reference parameters frozen.

print(
    "Policy parameters:",
    sum(p.numel() for p in policy_model.parameters())
)

print(
    "Reference parameters:",
    sum(p.numel() for p in ref_model.parameters())
)

print(
    "Trainable policy parameters:",
    sum(
        p.numel()
        for p in policy_model.parameters()
        if p.requires_grad
    )
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Policy parameters: 124439808
Reference parameters: 124439808
Trainable policy parameters: 124439808


In [ ]:
# Configure DPOTrainer

"""A few explanations:
- beta=0.1: controls how strongly DPO is regularized relative to the reference policy.
- load_best_model_at_end=True: after training, the trainer restores the checkpoint with the lowest validation DPO loss.
- save_total_limit=3: avoids accumulating too many large checkpoints.
- remove_unused_columns=False: preserves the preference dataset fields for DPO processing.
- fp16=True on your T4: uses its half-precision capability to reduce memory consumption and improve throughput.
- gradient_checkpointing=False: GPT-2 124M is small enough that we can initially avoid the additional recomputation overhead.

One important detail: MAX_LENGTH=160 is the combined prompt-and-completion limit, while MAX_PROMPT_LENGTH=120 limits the prompt portion.
This accommodates the prompt and generated continuation lengths from our R1 experiment.

"""

training_args = DPOConfig(
    # Output/checkpoint directory
    output_dir=str(OUTPUT_DIR),

    # Training duration
    num_train_epochs=NUM_TRAIN_EPOCHS,

    # Batch sizes
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Optimizer
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    #warmup_ratio=WARMUP_RATIO,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    max_grad_norm=1.0,

    # DPO
    beta=DPO_BETA,
    loss_type="sigmoid",

    # Sequence lengths
    max_length=MAX_LENGTH,
    #max_prompt_length is removed from recent versions of TRL
    #max_prompt_length=MAX_PROMPT_LENGTH,

    # Evaluation
    eval_strategy="steps",
    eval_steps=EVAL_SAVE_STEPS,

    # Checkpoint saving
    save_strategy="steps",
    save_steps=EVAL_SAVE_STEPS,
    save_total_limit=3,

    # Automatically restore the best evaluation checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging and tqdm
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    disable_tqdm=False,
    report_to="none",

    # GPU configuration
    fp16=USE_FP16,
    bf16=False,

    # Other
    gradient_checkpointing=False,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
)

In [ ]:
# Construct DPOTrainer

trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=validation_dataset,

    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/1178 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1178 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/1178 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/215 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/215 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/215 [00:00<?, ? examples/s]

At this point, TRL prepares the preference data, tokenizes the prompt/chosen/rejected sequences, and prepares the DPO training machinery.

In [ ]:
# Inspect the trainer
print("Trainer created.")
print("Training dataset:", len(trainer.train_dataset))
print("Evaluation dataset:", len(trainer.eval_dataset))
print("Output directory:", trainer.args.output_dir)

Trainer created.
Training dataset: 1178
Evaluation dataset: 215
Output directory: r2_checkpoints/runs/run_02


In [ ]:
# Detect existing checkpoints and train
# This is where we implement automatic checkpoint recovery.
"""The important behavior is:
- If Colab disconnects during training, the next execution detects the latest checkpoint and resumes.
- The model weights, optimizer state, scheduler state, and training progress are restored.
- It does not restart from R1 and repeat the earlier training steps.
This is why we use trainer.train(resume_from_checkpoint=...) instead of merely loading a model with from_pretrained().
The latter restores model weights but not the complete training state.
"""

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))

if last_checkpoint is not None:
    print("=" * 70)
    print("EXISTING CHECKPOINT DETECTED")
    print("=" * 70)

    print("Checkpoint:", last_checkpoint)
    print("Resuming training from checkpoint...")
    print()

    train_result = trainer.train(
        resume_from_checkpoint=last_checkpoint
    )

else:
    print("=" * 70)
    print("NO EXISTING CHECKPOINT FOUND")
    print("=" * 70)

    print("Starting a new DPO training run...")
    print()

    train_result = trainer.train()

NO EXISTING CHECKPOINT FOUND
Starting a new DPO training run...



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.588024,0.664097,3.620504,24195.000000,4.135742,6.518838,0.412605,-0.054384,-0.125040,0.635802,0.070656,-20.800460,-22.114530
100,0.549364,0.662673,3.397732,47707.000000,17.760991,20.666206,0.391644,-0.140645,-0.231960,0.626543,0.091315,-21.663063,-23.183725
150,0.433031,0.674589,3.026387,71581.000000,28.846882,32.053008,0.372519,-0.312081,-0.426676,0.598765,0.114595,-23.377424,-25.130886
200,0.402837,0.683661,2.696809,95699.000000,32.600611,35.141314,0.363130,-0.452487,-0.587177,0.604938,0.134690,-24.781486,-26.735900
250,0.401450,0.686910,2.512563,119497.000000,31.856416,34.047071,0.355890,-0.535364,-0.687133,0.623457,0.151769,-25.610258,-27.735455
300,0.315807,0.684328,2.480598,143348.000000,30.873809,33.049640,0.353003,-0.548665,-0.708577,0.628086,0.159912,-25.743260,-27.949897
350,0.288435,0.693010,2.382748,167629.000000,31.383964,33.628892,0.343118,-0.635076,-0.803471,0.623457,0.168395,-26.607378,-28.898842
400,0.350063,0.696603,2.342959,191270.000000,31.659809,33.909360,0.337856,-0.673165,-0.845630,0.623457,0.172465,-26.988261,-29.320426
444,0.402570,0.695891,2.341707,211944.000000,31.540820,33.794316,0.337033,-0.671433,-0.845197,0.618827,0.173764,-26.970945,-29.316095


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [ ]:
# One nuance: the actual optimizer-step count can vary slightly depending on
# how the Trainer handles the final incomplete gradient-accumulation group.
# We can inspect the actual planned number after creating the trainer:
print("Total training steps:", trainer.state.max_steps)
print("Warmup steps:", trainer.args.warmup_steps)

# TIP: if we want to preserve the original 5% warmup proportion when changing dataset size,
# recalculate it from the planned total steps as follows:
# warmup_steps = round(trainer.state.max_steps * 0.05)

Total training steps: 444
Warmup steps: 22


In [ ]:
# Save the best model
# Because we enabled load_best_model_at_end, the trainer should now have the best validation checkpoint loaded into its policy model.
# Save it as a standalone model

BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

print("=" * 70)
print("BEST MODEL SAVED")
print("=" * 70)

print("Model directory:", BEST_MODEL_DIR)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best validation loss:",
    trainer.state.best_metric
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BEST MODEL SAVED
Model directory: r2_checkpoints/r2-dpo-best
Best checkpoint: r2_checkpoints/runs/run_02/checkpoint-100
Best validation loss: 0.6626730561256409


In [ ]:
# Save training metrics

train_metrics = train_result.metrics

trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)

eval_metrics = trainer.evaluate()

trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)

print("\nFinal evaluation metrics:")
print(eval_metrics)

***** train metrics *****
  epoch                    =        3.0
  total_flos               =   135080GF
  train_loss               =     0.4495
  train_runtime            = 0:03:50.03
  train_samples_per_second =     15.363
  train_steps_per_second   =       1.93


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.402570,0.662673,444,3.397732,211944.000000,17.760991,20.666206,0.391644,-0.140645,-0.231960,0.626543,0.091315,-21.663063,-23.183725


***** eval metrics *****
  eval_entropy             =   3.3977
  eval_logits/chosen       =   17.761
  eval_logits/rejected     =  20.6662
  eval_logps/chosen        = -21.6631
  eval_logps/rejected      = -23.1837
  eval_loss                =   0.6627
  eval_mean_token_accuracy =   0.3916
  eval_num_tokens          = 211944.0
  eval_rewards/accuracies  =   0.6265
  eval_rewards/chosen      =  -0.1406
  eval_rewards/margins     =   0.0913
  eval_rewards/rejected    =   -0.232

Final evaluation metrics:
{'eval_loss': 0.6626730561256409, 'eval_entropy': 3.3977321518792047, 'eval_num_tokens': 211944.0, 'eval_logits/chosen': 17.760991048864327, 'eval_logits/rejected': 20.66620623308032, 'eval_mean_token_accuracy': 0.3916443393186287, 'eval_rewards/chosen': -0.14064476894283737, 'eval_rewards/rejected': -0.2319597087248608, 'eval_rewards/accuracies': 0.6265432100605082, 'eval_rewards/margins': 0.09131493671210827, 'eval_logps/chosen': -21.663062961013228, 'eval_logps/rejected': -23.18372459

The resulting directory structure should look like this:
```r2_checkpoints/
│
├── runs/
│   └── run_01/
│       ├── checkpoint-50/
│       ├── checkpoint-100/
│       ├── checkpoint-150/
│       └── ...
│
└── r2-dpo-best/
    ├── config.json
    ├── generation_config.json
    ├── model.safetensors
    ├── tokenizer.json
    ├── tokenizer_config.json
    └── ...
```
The exact checkpoint numbers depend on the number of training batches and epochs.

**How to resume and continue training**

This is worth understanding carefully because it will be useful in our later PPO and GRPO experiments too.

**Scenario A: Colab disconnects during training**

Simply rerun the notebook from the beginning, retaining the same:
```python
RUN_NAME = "run_01"
INITIAL_POLICY_PATH = R1_CHECKPOINT
```

The code detects the latest `checkpoint-N` under `runs/run_01` and resumes training.

Don't delete that directory.

**Scenario B: R2 finishes, and you want to improve it further**

Suppose R2 completes three epochs, and you want to train for another two epochs.

Change the configuration to:
```python
RUN_NAME = "run_02"
# Start from the best R2 model.
INITIAL_POLICY_PATH = str(BEST_MODEL_DIR)
# Keep the reference model unchanged.
REFERENCE_MODEL_PATH = R1_CHECKPOINT
# Train for another two epochs.
NUM_TRAIN_EPOCHS = 2
```

The new run will use:
```
r2_checkpoints/
    runs/
        run_01/
        run_02/
    r2-dpo-best/
```

This is a new training phase:

- Policy starts from the best `R2` model.
- Reference remains the original `R1` model.
- A new optimizer and scheduler are initialized.
- New checkpoints are saved under `run_02`.

This is a legitimate way to continue DPO optimization without accidentally reusing an exhausted learning-rate schedule.

**Important**: Don't change `RUN_NAME` when recovering an interrupted run. Only change it when deliberately starting a new training phase.

In [ ]:
# Save preference dataset to Google Drive

!cp -r ./r2_preference_dataset drive/MyDrive/datasets

# Save r2 to Google Drive

# If we have enough storage to use on our Google Drive, save checkpoints and final model
!cp -r ./r2_checkpoints drive/MyDrive/checkpoints

# Otherwise, save final model
!cp -r ./r2_checkpoints/r2-dpo-best drive/MyDrive/checkpoints/r2_checkpoints

In [ ]:
# Load R2 for inference
# After training, load the best checkpoint

from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
)

R2_MODEL_PATH = "./r2_checkpoints/r2-dpo-best"

r2_tokenizer = AutoTokenizer.from_pretrained(
    R2_MODEL_PATH
)

r2_tokenizer.pad_token = r2_tokenizer.eos_token

r2_model = GPT2LMHeadModel.from_pretrained(
    R2_MODEL_PATH
)

r2_model.to(DEVICE)
r2_model.eval()

r2_model.config.use_cache = True

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
# A generation function compatible with our R1 experiment

@torch.inference_mode()
def generate_r2_continuation(
    prefix,
    sentiment,
    max_new_tokens=40,
):
    prompt = (
        f"Sentiment: {sentiment}\n"
        f"Continue the following review:\n"
        f"{prefix}"
    )

    inputs = r2_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)

    input_length = inputs["input_ids"].shape[1]

    outputs = r2_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=r2_tokenizer.pad_token_id,
        eos_token_id=r2_tokenizer.eos_token_id,
    )

    continuation = r2_tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return continuation

In [ ]:
# Test it
prefix = "that loves its characters and communicates something"

print("Positive:")
print(generate_r2_continuation(prefix, "positive"))

print("\nNegative:")
print(generate_r2_continuation(prefix, "negative"))

Positive:
 very special about friendship

Negative:
 disorienting about human nature


In [ ]:
# Compare to R1

# R1 generation

def generate_continuation_R1(
    model,
    tokenizer,
    sentiment,
    prefix,
    max_new_tokens=40,
):

    prompt = (
        f"Sentiment: {sentiment}\n"
        f"Continue the following review:\n"
        f"{prefix}"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(DEVICE)

    model.eval()

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        0, inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

ref_tokenizer = AutoTokenizer.from_pretrained(
    REFERENCE_MODEL_PATH
)

ref_model.eval()
ref_model.config.use_cache = True

for sentiment in ["positive", "negative"]:

    continuation = generate_continuation_R1(
        ref_model,
        ref_tokenizer,
        sentiment,
        prefix,
    )

    print(f"\nSentiment: {sentiment}")
    print(f"{prefix}{continuation}")



Sentiment: positive
that loves its characters and communicates something beautiful about the human condition

Sentiment: negative
that loves its characters and communicates something about itself without giving it a real sense of emotional depth


In [ ]:
# A common continuation generation function to be used for all experiments (models), making a comparison across them fairer

def generate_continuation(
    model,
    tokenizer,
    sentiment,
    prefix,
    max_new_tokens=40
):
    """
    Generate a sentiment-conditioned continuation.

    Args:
        model: Hugging Face causal language model.
        tokenizer: Corresponding tokenizer.
        sentiment: "positive" or "negative".
        prefix: Review prefix to continue.
        max_new_tokens: Maximum number of new tokens.

    Returns:
        Generated continuation (without the original prompt).
    """

    # Construct prompt exactly as in R1 and R2.
    prompt = (
        f"Sentiment: {sentiment}\n"
        f"Continue the following review:\n"
        f"{prefix}"
    )

    # Determine model device.
    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    model.eval()

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Remove prompt tokens to obtain continuation only.
    prompt_length = inputs["input_ids"].shape[1]

    continuation_ids = output_ids[0, prompt_length:]

    continuation = tokenizer.decode(
        continuation_ids,
        skip_special_tokens=True
    )

    return continuation

In [ ]:
# Compare to baseline R0 (vanilla GPT-2)

MODEL_NAME = "openai-community/gpt2"

GPT2_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no dedicated padding token.
GPT2_tokenizer.pad_token = GPT2_tokenizer.eos_token

GPT2_model = GPT2LMHeadModel.from_pretrained(
    MODEL_NAME
)

GPT2_model.config.pad_token_id = GPT2_tokenizer.pad_token_id
GPT2_model.config.use_cache = True

GPT2_model.eval()
GPT2_model.to(DEVICE)

for sentiment in ["positive", "negative"]:

    continuation = generate_continuation(
        model=GPT2_model,
        tokenizer=GPT2_tokenizer,
        sentiment=sentiment,
        prefix=prefix,
        max_new_tokens=40
    )

    print(f"\nR0 - {sentiment.capitalize()}:")
    print(continuation)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


R0 - Positive:
 of a quality to them, but in my experience there are times where it's just not going for me. It feels like that sort Of Heroine-esque plot is being told as an act of

R0 - Negative:
 of an emotional state, I am very pleased with this story. (In case you missed it :I will continue to write reviews for each issue until a later date.)This is by far one that


In [ ]:
# Let's use the same generation_continuation function for R0 (already done above), R1, and R2
for sentiment in ["positive", "negative"]:

    GPT2_continuation = generate_continuation(
        model=GPT2_model,
        tokenizer=GPT2_tokenizer,
        sentiment=sentiment,
        prefix=prefix,
        max_new_tokens=40
    )

    print(f"\nR0 - {sentiment.capitalize()}:")
    print(GPT2_continuation)

    R1_continuation = generate_continuation(
        model=ref_model,
        tokenizer=ref_tokenizer,
        sentiment=sentiment,
        prefix=prefix,
        max_new_tokens=40
    )

    print(f"\nR1 - {sentiment.capitalize()}:")
    print(R1_continuation)

    R2_continuation = generate_continuation(
        model=r2_model,
        tokenizer=r2_tokenizer,
        sentiment=sentiment,
        prefix=prefix,
        max_new_tokens=40
    )

    print(f"\nR2 - {sentiment.capitalize()}:")
    print(R2_continuation)


R0 - Positive:
 in a certain way that is different than what you'd think. I loved it to pieces, so many things are about them - like how much he has changed since his time with this ship (if

R1 - Positive:
 special

R2 - Positive:
 uniquely magical

R0 - Negative:
 new. i think they were trying to be funny but also not at all like a straight up lesbian comic that has people in it who are gay, or just really hot guys as opposed of being super

R1 - Negative:
 rather weird and sad about human nature

R2 - Negative:
 offensive about women


**What to monitor during training**

TRL's DPOTrainer provides more interesting metrics than just training loss.

Interesting things to inspect:

**Metric**
```
loss                DPO training objective
eval_loss           DPO objective on held-out preference pairs
rewards/chosen      Average implicit reward of chosen responses
rewards/rejected    Average implicit reward of rejected responses
rewards/margins     Difference between chosen and rejected implicit rewards
rewards/accuracies  Fraction of pairs where chosen has higher implicit reward
logps/chosen        Policy log probability of chosen continuations
logps/rejected      Policy log probability of rejected continuations
```

The most interesting ones for understanding DPO are: rewards/margins and rewards/accuracies.

We want the policy to increasingly distinguish chosen responses from rejected responses.

Don't assume that lower training loss automatically means better generated text. We need to evaluate actual sentiment adherence and text quality as well.